# Turkish Legal RAG - Qwen3 Embedding Evaluation

Bu notebook embedding fine-tuning yapmaz. Amaç, mevcut BGE-M3 embedding baseline yerine daha güçlü bir hazır embedding modeli denemektir.

Denenen model:

```text
Qwen/Qwen3-Embedding-0.6B
```

Notebook şunları yapar:

1. Kaggle datasetindeki corpus, benchmark ve scriptleri working klasörüne kopyalar.
2. Qwen3 embedding modeliyle corpus'u yeniden vektörleştirir.
3. Yeni FAISS index kurar.
4. Gold benchmark üzerinde dense ve hybrid retrieval skorlarını ölçer.
5. Mevcut BGE-M3 hybrid baseline ile karşılaştırır.

In [ ]:
!nvidia-smi

## 1. Paketleri Kur

In [ ]:
!pip install -q -U "sentence-transformers>=3.0.0" "transformers>=4.51.0" accelerate faiss-cpu tqdm

## 2. Dosyaları Kaggle Working Klasörüne Kopyala

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/index_qwen3",
    WORK_DIR / "data/eval",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

for name in [
    "build_faiss_index.py",
    "evaluate_retrieval.py",
    "search_faiss.py",
    "search_hybrid.py",
]:
    copy_required(name, WORK_DIR / "scripts")

for name in ["retrieval_chunks.json", "retrieval_corpus.json"]:
    copy_required(name, WORK_DIR / "data/processed")

copy_required("qa_benchmark_gold.csv", WORK_DIR / "data/eval")

# Existing BGE-M3 baseline index. This is only used for quick comparison.
for name in ["faiss_bge_m3.index", "metadata_bge_m3.json", "index_config_bge_m3.json"]:
    copy_required(name, WORK_DIR / "data/index")

print("\nWorking files:")
!find /kaggle/working/legal-rag -maxdepth 4 -type f | sort

## 3. Qwen3 Embedding Ayarları

In [ ]:
from pathlib import Path

WORK_DIR = Path("/kaggle/working/legal-rag")
QWEN_MODEL = "Qwen/Qwen3-Embedding-0.6B"
QUERY_PROMPT = (
    "Instruct: Given a Turkish legal question, retrieve the relevant Turkish law article "
    "or statute passage that answers the question\nQuery: "
)

QWEN_INDEX = WORK_DIR / "data/index_qwen3/faiss_qwen3_embedding_06b.index"
QWEN_METADATA = WORK_DIR / "data/index_qwen3/metadata_qwen3_embedding_06b.json"
QWEN_CONFIG = WORK_DIR / "data/index_qwen3/index_config_qwen3_embedding_06b.json"

print("Model:", QWEN_MODEL)
print("Query prompt:", QUERY_PROMPT)

## 4. Qwen3 ile FAISS Index Kur

In [ ]:
import subprocess

cmd = [
    "python", str(WORK_DIR / "scripts/build_faiss_index.py"),
    "--chunks", str(WORK_DIR / "data/processed/retrieval_chunks.json"),
    "--index-out", str(QWEN_INDEX),
    "--metadata-out", str(QWEN_METADATA),
    "--config-out", str(QWEN_CONFIG),
    "--model", QWEN_MODEL,
    "--device", "cuda",
    "--batch-size", "4",
    "--passage-prefix", "",
    "--query-prompt", QUERY_PROMPT,
    "--max-seq-length", "2048",
]
subprocess.run(cmd, check=True)

!ls -lh /kaggle/working/legal-rag/data/index_qwen3/

## 5. BGE-M3 Baseline Hybrid'i Tekrar Ölç

In [ ]:
import subprocess

cmd = [
    "python", str(WORK_DIR / "scripts/evaluate_retrieval.py"),
    "--benchmark", str(WORK_DIR / "data/eval/qa_benchmark_gold.csv"),
    "--corpus", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
    "--chunks", str(WORK_DIR / "data/processed/retrieval_chunks.json"),
    "--index", str(WORK_DIR / "data/index/faiss_bge_m3.index"),
    "--metadata", str(WORK_DIR / "data/index/metadata_bge_m3.json"),
    "--config", str(WORK_DIR / "data/index/index_config_bge_m3.json"),
    "--mode", "hybrid",
    "--embedding-device", "cuda",
    "--top-k", "10",
    "--output", str(WORK_DIR / "data/eval/eval_hybrid_bge_m3_recheck.json"),
]
subprocess.run(cmd, check=True)

## 6. Qwen3 Dense Retrieval Evaluation

In [ ]:
import subprocess

cmd = [
    "python", str(WORK_DIR / "scripts/evaluate_retrieval.py"),
    "--benchmark", str(WORK_DIR / "data/eval/qa_benchmark_gold.csv"),
    "--corpus", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
    "--chunks", str(WORK_DIR / "data/processed/retrieval_chunks.json"),
    "--index", str(QWEN_INDEX),
    "--metadata", str(QWEN_METADATA),
    "--config", str(QWEN_CONFIG),
    "--mode", "dense",
    "--embedding-device", "cuda",
    "--embedding-batch-size", "4",
    "--top-k", "10",
    "--output", str(WORK_DIR / "data/eval/eval_dense_qwen3_embedding_06b.json"),
]
subprocess.run(cmd, check=True)

## 7. Qwen3 + BM25 Hybrid Evaluation

In [ ]:
import subprocess

cmd = [
    "python", str(WORK_DIR / "scripts/evaluate_retrieval.py"),
    "--benchmark", str(WORK_DIR / "data/eval/qa_benchmark_gold.csv"),
    "--corpus", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
    "--chunks", str(WORK_DIR / "data/processed/retrieval_chunks.json"),
    "--index", str(QWEN_INDEX),
    "--metadata", str(QWEN_METADATA),
    "--config", str(QWEN_CONFIG),
    "--mode", "hybrid",
    "--embedding-device", "cuda",
    "--embedding-batch-size", "4",
    "--top-k", "10",
    "--output", str(WORK_DIR / "data/eval/eval_hybrid_qwen3_embedding_06b.json"),
]
subprocess.run(cmd, check=True)

## 8. Sonuçları Yan Yana Göster

In [ ]:
import json
from pathlib import Path

result_files = [
    WORK_DIR / "data/eval/eval_hybrid_bge_m3_recheck.json",
    WORK_DIR / "data/eval/eval_dense_qwen3_embedding_06b.json",
    WORK_DIR / "data/eval/eval_hybrid_qwen3_embedding_06b.json",
]

for path in result_files:
    print("\n" + "=" * 100)
    print(path.name)
    print("=" * 100)
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data["summary"], ensure_ascii=False, indent=2))

## 9. Örnek Soru Aramaları

In [ ]:
import subprocess

def run_qwen_hybrid(question: str):
    print("=" * 120)
    print(question)
    print("=" * 120)
    cmd = [
        "python", str(WORK_DIR / "scripts/search_hybrid.py"),
        question,
        "--top-k", "5",
        "--index", str(QWEN_INDEX),
        "--metadata", str(QWEN_METADATA),
        "--config", str(QWEN_CONFIG),
        "--articles", str(WORK_DIR / "data/processed/retrieval_corpus.json"),
        "--device", "cuda",
        "--no-query-expansion",
        "--show-text",
    ]
    subprocess.run(cmd, check=True)

for question in [
    "işçi 2 gün işe gelmezse ne olur?",
    "kişisel veriler yurt dışına hangi şartlarda aktarılır?",
    "birini öldürmek suç mudur?",
]:
    run_qwen_hybrid(question)

## 10. Kaydedilecek Dosyalar

Kaggle çıktılarını kaydetmek istersen özellikle şu klasör/dosyalar önemli:

```text
/kaggle/working/legal-rag/data/index_qwen3/
/kaggle/working/legal-rag/data/eval/eval_dense_qwen3_embedding_06b.json
/kaggle/working/legal-rag/data/eval/eval_hybrid_qwen3_embedding_06b.json
/kaggle/working/legal-rag/data/eval/eval_hybrid_bge_m3_recheck.json
```

Eğer Qwen3 hybrid skoru BGE-M3 hybrid skorunu geçerse final retrieval modelini Qwen3 olarak değiştirebiliriz. Geçmezse bu deney ablation tablosuna eklenir.